# DemandIQ: Exploratory Data Analysis & Demand Profiling 📊

**Project:** DemandIQ — AI Demand Forecasting & Inventory Intelligence  
**Objective:** Comprehensive exploratory analysis of the Kaggle Food Demand Forecasting dataset across 77 fulfillment centers and 51 meal options over 145 weeks to uncover demand dynamics, price elasticity, promotional uplift, and temporal seasonality.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from demandiq.data import load_raw_datasets
from demandiq.pipeline import generate_synthetic_sample_data

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("DemandIQ EDA environment initialized.")

## 1. Data Ingestion & Integrity Validation
Loading raw dataset from `data/raw/` or fallback sample generator.

In [ ]:
try:
    datasets = load_raw_datasets()
    print("Loaded production raw CSVs from data/raw/")
except Exception as e:
    print(f"Raw CSVs not present ({e}). Using synthetic benchmark sample.")
    datasets = generate_synthetic_sample_data(num_weeks=145, num_centers=10, num_meals=15)

train_df = datasets["train"]
centers_df = datasets["centers"]
meals_df = datasets["meals"]

# Merge datasets into unified analysis DataFrame
df = train_df.merge(centers_df, on="center_id", how="left").merge(
    meals_df, on="meal_id", how="left"
)
print(f"Unified dataset shape: {df.shape}")
df.head()

## 2. Target Distribution & Log-Transformation Analysis
Food order demand is typically heavily right-skewed. We inspect the raw vs log-transformed distribution to justify optimizing with RMSLE.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Raw Demand Distribution
sns.histplot(df["num_orders"], kde=True, ax=axes[0], color="royalblue", bins=40)
axes[0].set_title(f"Raw Demand Distribution (Skew: {df['num_orders'].skew():.2f})")
axes[0].set_xlabel("Number of Orders (num_orders)")

# Log-Transformed Demand
log_orders = np.log1p(df["num_orders"])
sns.histplot(log_orders, kde=True, ax=axes[1], color="seagreen", bins=40)
axes[1].set_title(f"Log-Transformed Demand: log1p(num_orders) (Skew: {log_orders.skew():.2f})")
axes[1].set_xlabel("log1p(num_orders)")

plt.tight_layout()
plt.show()

## 3. Price Elasticity & Promotional Uplift Analysis
Analyzing discount depth (`base_price - checkout_price`) and marketing promotion impact on sales velocity.

In [ ]:
df["discount_ratio"] = (df["base_price"] - df["checkout_price"]) / df["base_price"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Price vs Demand Scatter
sns.scatterplot(
    data=df.sample(min(1000, len(df))),
    x="checkout_price",
    y="num_orders",
    hue="category",
    alpha=0.7,
    ax=axes[0],
)
axes[0].set_title("Price Elasticity: Checkout Price vs Demand")

# Promotional Uplift by Email & Homepage Feature
promo_summary = (
    df.groupby(["emailer_for_promotion", "homepage_featured"])["num_orders"].mean().reset_index()
)
sns.barplot(
    data=promo_summary,
    x="emailer_for_promotion",
    y="num_orders",
    hue="homepage_featured",
    ax=axes[1],
    palette="viridis",
)
axes[1].set_title("Promotional Uplift: Average Orders by Campaign Type")
axes[1].set_xlabel("Emailer for Promotion (0/1)")
axes[1].set_ylabel("Mean Order Volume")

plt.tight_layout()
plt.show()

## 4. Time Series Trend & Seasonality Decomposition
Inspecting aggregate weekly demand trajectory over 145 weeks.

In [ ]:
weekly_aggregate = df.groupby("week")["num_orders"].sum().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(
    weekly_aggregate["week"],
    weekly_aggregate["num_orders"],
    color="indigo",
    linewidth=2,
    marker="o",
    markersize=3,
)
plt.title("Total Weekly Demand Trajectory Across Network (Weeks 1 to 145)")
plt.xlabel("Week Number")
plt.ylabel("Total Network Orders")
plt.axvline(x=125, color="red", linestyle="--", label="Train / Val Split (Week 125)")
plt.axvline(x=135, color="orange", linestyle="--", label="Val / Test Split (Week 135)")
plt.legend()
plt.show()

## 5. Key EDA Findings & Pipeline Implications
1. **Log Target Transformation:** Right-skewness confirms models must optimize `log1p(num_orders)` to directly minimize RMSLE and avoid scale distortion.
2. **Strong Promotional Uplift:** Synergistic spikes occur when both homepage featured and email campaigns align.
3. **Dynamic Safety Buffers Needed:** Granular volatility varies by SKU; high-velocity categories need adaptive safety stock to prevent stockouts.